In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [4]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [6]:
train_x = train.drop(columns=['id', 'efficiency'])

train_y = train['efficiency']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [8]:
kf = KFold(n_splits=5)

ExtraT_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    ExtraT = ExtraTreesRegressor(n_estimators=50, max_depth=8, min_samples_split=2, min_samples_leaf=2, random_state=42)
    ExtraT.fit(x_fold, y_fold)

    #ExtraT_pre_test += ExtraT.predict(test)

    print(ExtraT.score(x_test_fold.to_numpy(), y_test_fold.to_numpy()))

FOLD: 0
0.4498277211721301
FOLD: 1
0.5028664202403552
FOLD: 2
0.49631771675070224
FOLD: 3
0.46886526783417615
FOLD: 4
0.43865143386946415


In [9]:
test

,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type
0,17.618379,85.449838,10828.0,13.910963,6.0,0.889765,6.370396,0.069101,19.517274,33.509889,9668.0,10606.0,2.0,1.0,3.0
1,34.826323,722.801748,1674.0,20.916528,4.0,0.590372,30.095867,1.713852,37.421443,32.327060,7239.0,11031.0,3.0,0.0,0.0
2,33.776934,485.491998,6159.0,1.446962,3.0,0.611425,28.424430,1.696936,32.147763,69.613333,8942.0,11782.0,3.0,1.0,1.0
3,18.584189,350.022720,5249.0,18.810133,5.0,0.700468,7.848038,0.787188,25.734118,42.862760,6200.0,9771.0,2.0,2.0,1.0
4,43.044908,437.295622,9384.0,17.473594,8.0,0.564938,12.300717,1.867620,30.038138,51.025763,3096.0,3686.0,1.0,3.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,40.581656,492.446562,3537.0,4.508170,8.0,0.700468,40.883724,3.434505,48.476050,14.015174,6468.0,2851.0,3.0,0.0,3.0
11996,16.958524,198.844667,4095.0,4.021203,4.0,0.810999,0.000000,1.290352,23.502657,22.788465,11408.0,9030.0,2.0,2.0,3.0
11997,24.055333,757.621634,1545.0,15.253932,3.0,0.700468,5.855590,4.835729,31.908375,98.197373,5048.0,4093.0,0.0,0.0,0.0
11998,15.623725,177.376256,90.0,16.437613,2.0,0.861087,0.000000,1.159060,23.835495,9.581184,9499.0,6844.0,1.0,3.0,1.0


In [10]:
cat_pre_test=ExtraT.predict(test)
print(f'Concordance index (lifelines): {mean_absolute_error(y_test, ExtraT.predict(X_test))}')

Concordance index (lifelines): 0.0512155566544232


In [11]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = (cat_pre_test)
test_predictions

array([0.40111288, 0.54237062, 0.51452773, ..., 0.58762577, 0.41605131,
       0.54213066])

In [12]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.401113
1,1,0.542371
2,2,0.514528
3,3,0.462815
4,4,0.468096
...,...,...
11995,11995,0.534823
11996,11996,0.450877
11997,11997,0.587626
11998,11998,0.416051


In [13]:
submission.to_csv('submission.csv', index = False)